# Headout EN → FR  ·  NLLB-200-600M + QLoRA (CPU/GPU)

**Model** : `facebook/nllb-200-distilled-600M` (~600M params)  
**Method** : QLoRA on GPU (4-bit NF4) · Regular LoRA on CPU (fp32)  
**Export** : ~1.2 GB merged (vs 4.7 GB for NLLB-1.3B)  

### GPU vs CPU behaviour
- **GPU** : 4-bit QLoRA + `paged_adamw_8bit` + bf16 — same as 1.3B notebook
- **CPU** : full fp32 LoRA, no quantization, `adamw_torch` — much slower but works

## 1 · GPU Check

In [ ]:
import os, subprocess, torch

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU / nvidia-smi not found')

USE_GPU = torch.cuda.is_available()
DEVICE  = 'cuda' if USE_GPU else 'cpu'

if USE_GPU:
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('Mode : GPU  |  QLoRA (4-bit) enabled  |  TF32 ✓')
else:
    print('Mode : CPU  |  regular LoRA (fp32) — bitsandbytes 4-bit requires CUDA')
    print('WARNING: training 600M params on CPU is ~15-20× slower than GPU')

## 2 · Install Dependencies

In [ ]:
!pip install -q \
    'transformers>=4.40.0' \
    'datasets>=2.19.0' \
    'peft>=0.10.0' \
    'bitsandbytes>=0.43.0' \
    'accelerate>=0.29.0' \
    'sacrebleu>=2.4.0' \
    'sacremoses>=0.1.1' \
    'sentencepiece>=0.1.99' \
    'tensorboard>=2.16.0'
print('Done.')

## 3 · Config

In [ ]:
import math
from pathlib import Path

MODEL_NAME = 'facebook/nllb-200-distilled-600M'
SRC_LANG   = 'eng_Latn'
TGT_LANG   = 'fra_Latn'
MAX_LEN    = 64

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# Same as NLLB-1.3B notebook — batch=32 logits are only 1.05 GB on T4 (fits easily)
if USE_GPU:
    TRAIN_BATCH = 32
    GRAD_ACCUM  = 4     # effective batch = 32 × 4 = 128
    EVAL_BATCH  = 32
else:
    TRAIN_BATCH = 4
    GRAD_ACCUM  = 32    # effective batch = 4 × 32 = 128
    EVAL_BATCH  = 4

LEARNING_RATE = 2e-4
WARMUP_STEPS  = 100
NUM_EPOCHS    = 3
EVAL_STEPS    = 200
ES_PATIENCE   = 5

OUT_DIR      = './checkpoints-nllb600'
ADAPTER_DIR  = './lora-adapter-nllb600'
BEST_DIR     = f'{OUT_DIR}/best'

train_size      = 90_000
steps_epoch     = math.ceil(train_size / TRAIN_BATCH)
optimizer_steps = math.ceil(steps_epoch / GRAD_ACCUM) * NUM_EPOCHS
print(f'Device             : {DEVICE}')
print(f'Train batch        : {TRAIN_BATCH}  ×  grad_accum {GRAD_ACCUM}  =  {TRAIN_BATCH*GRAD_ACCUM} effective')
print(f'Raw steps/epoch    : {steps_epoch:,}')
print(f'Optimizer updates  : {optimizer_steps:,}')
logit_gb = TRAIN_BATCH * MAX_LEN * 256_206 * 2 / 1e9
print(f'Logits per step    : {logit_gb:.2f} GB')
print(f'Est. time on T4    : ~{optimizer_steps / 0.4 / 3600:.1f} hrs')

## 4 · Upload Pre-Parsed JSON Files

Same format as the NLLB notebook: `train.json`, `val.json`, `test.json`  
Each is `[{"en": "...", "fr": "..."}]`

In [ ]:
import os, json
from pathlib import Path

Path('data').mkdir(exist_ok=True)

missing = [s for s in ['train', 'val', 'test'] if not Path(f'data/{s}.json').exists()]
if missing:
    try:
        from google.colab import files
        print(f'Upload {missing} JSON files:')
        uploaded = files.upload()
        for name, content in uploaded.items():
            with open(f'data/{name}', 'wb') as f:
                f.write(content)
            print(f'  data/{name}  ({len(content)/1e6:.1f} MB)')
    except ImportError:
        raise FileNotFoundError(
            f'Missing: {missing}. Upload via Workbench file browser '
            f'or gsutil cp gs://YOUR_BUCKET/{{train,val,test}}.json data/'
        )

for split in ['train', 'val', 'test']:
    p = Path(f'data/{split}.json')
    with open(p) as f:
        n = len(json.load(f))
    print(f'data/{split}.json  →  {n:,} pairs  ({p.stat().st_size/1e6:.1f} MB)')

## 5 · Placeholder Normalization

Same as NLLB notebook — XML `<ph>` → `[PH_N]`, URLs normalized, curly keys kept as-is.

In [ ]:
import re

_PH_XML   = re.compile(r'<ph\b[^>]*/></ph>|<ph\b[^>]*>.*?</ph>', re.DOTALL)
_PH_IDX   = re.compile(r'x="(\d+)"')
_BPT      = re.compile(r'<bpt\b[^>]*>.*?</bpt>', re.DOTALL)
_EPT      = re.compile(r'<ept\b[^>]*>.*?</ept>', re.DOTALL)
_TAG_IDX  = re.compile(r'i="(\d+)"')
_X_TAG    = re.compile(r'<x\s+id="x(\d+)"\s*/>')          # standalone <x id="x1"/> → [X_1]
_URL_TAG  = re.compile(r'<url_(\d+)>')                     # upstream <url_0> → [URLTAG_0]
_QUILL    = re.compile(r'<quillbot-extension-portal[^>]*>', re.IGNORECASE)  # strip browser artifact
_MD_LINK  = re.compile(r'\[([^\]]+)\]\s*\((https?://[^)]+)\)')
_BARE_URL = re.compile(r'https?://\S+')
_BROKEN   = re.compile(r'\{([^})]{1,40}?)\)')

CURLY_FR_TO_EN = {
    'transferts': 'transfers', 'annulation': 'cancel',
    'horloge': 'clock',        'calendrier': 'calendar',
    'pone': 'phone',           "casques d'écoute": 'headphones',
    'casques': 'headphones',   'écouteurs': 'headphones',
}

def _fix_curly(text):
    text = _BROKEN.sub(lambda m: '{' + m.group(1) + '}', text)
    def _rep(m):
        key = m.group(1).strip().lower()
        return '{' + CURLY_FR_TO_EN.get(key, key) + '}'
    return re.sub(r"\"\{([a-zA-ZéàîôùèêâûœçÉÀÎÔÙÈÊÂÛŒÇ _']{1,40}?)}\"", _rep, text)

def normalize(text, sc):
    # Strip browser extension artifacts
    text = _QUILL.sub('', text)
    text = _fix_curly(text)

    # <ph> tags
    ph = {}
    def _r_ph(m):
        mi = _PH_IDX.search(m.group(0))
        idx = mi.group(1) if mi else str(len(ph))
        ph[idx] = m.group(0); return f'[PH_{idx}]'
    text = _PH_XML.sub(_r_ph, text)
    if ph: sc['ph'] = ph

    # <bpt> / <ept> tags
    bpt, ept = {}, {}
    def _r_bpt(m):
        mi = _TAG_IDX.search(m.group(0))
        idx = mi.group(1) if mi else str(len(bpt))
        bpt[idx] = m.group(0); return f'[BPT_{idx}]'
    def _r_ept(m):
        mi = _TAG_IDX.search(m.group(0))
        idx = mi.group(1) if mi else str(len(ept))
        ept[idx] = m.group(0); return f'[EPT_{idx}]'
    text = _BPT.sub(_r_bpt, text)
    text = _EPT.sub(_r_ept, text)
    if bpt: sc['bpt'] = bpt; sc['ept'] = ept

    # standalone <x id="xN"/> tags  ← NEW
    x_tags = {}
    def _r_x(m):
        idx = m.group(1); x_tags[idx] = m.group(0)
        return f'[X_{idx}]'
    text = _X_TAG.sub(_r_x, text)
    if x_tags: sc['x'] = x_tags

    # upstream <url_N> tags  ← NEW
    url_tags = {}
    def _r_url_tag(m):
        idx = m.group(1); url_tags[idx] = m.group(0)
        return f'[URLTAG_{idx}]'
    text = _URL_TAG.sub(_r_url_tag, text)
    if url_tags: sc['url_tags'] = url_tags

    # Markdown links and bare URLs
    urls = {}
    ctr  = [0]
    def _r_md(m):
        k = f'URL_{ctr[0]}'; urls[k] = m.group(2); ctr[0] += 1
        return f'[{m.group(1)}]({k})'
    text = _MD_LINK.sub(_r_md, text)
    def _r_url(m):
        idx = len(urls); urls[f'BARE_{idx}'] = m.group(0)
        return f'[URL_{idx}]'
    text = _BARE_URL.sub(_r_url, text)
    if urls: sc['urls'] = urls

    return re.sub(r'\]\s+\(', '](', text)

def restore(norm_fr, sc):
    for idx, orig in sc.get('ph',  {}).items(): norm_fr = norm_fr.replace(f'[PH_{idx}]',  orig)
    for idx, orig in sc.get('bpt', {}).items(): norm_fr = norm_fr.replace(f'[BPT_{idx}]', orig)
    for idx, orig in sc.get('ept', {}).items(): norm_fr = norm_fr.replace(f'[EPT_{idx}]', orig)
    for idx, orig in sc.get('x',   {}).items(): norm_fr = norm_fr.replace(f'[X_{idx}]',   orig)
    for idx, orig in sc.get('url_tags', {}).items(): norm_fr = norm_fr.replace(f'[URLTAG_{idx}]', orig)
    for key, url in sc.get('urls', {}).items():
        if key.startswith('BARE_'):
            norm_fr = norm_fr.replace(f'[URL_{key.split("_",1)[1]}]', url)
        else:
            norm_fr = norm_fr.replace(key, url)
    return norm_fr

def validate_ph(norm_en, norm_fr):
    pat = re.compile(r'\[(?:PH|BPT|EPT|X|URLTAG|URL)_[\w]+\]')
    return set(pat.findall(norm_en)) == set(pat.findall(norm_fr))

def repair_ph(norm_en, norm_fr):
    pat = re.compile(r'\[(?:PH|BPT|EPT|X|URLTAG|URL)_[\w]+\]')
    missing = [t for t in pat.findall(norm_en) if t not in norm_fr]
    if missing: norm_fr = norm_fr.rstrip() + ' ' + ' '.join(missing)
    return norm_fr

# Smoke tests
sc = {}
n = normalize('<ph x="0" type="x"><x/></ph>Skip the Line {clock}', sc)
assert '[PH_0]' in n and '{clock}' in n, 'ph test failed'

sc2 = {}
n2 = normalize('Book <x id="x1"/> tickets <x id="x2"/> now', sc2)
assert '[X_1]' in n2 and '[X_2]' in n2, 'x-tag test failed'
assert restore(n2, sc2) == 'Book <x id="x1"/> tickets <x id="x2"/> now', 'restore test failed'

sc3 = {}
n3 = normalize('Visit <url_0> for details', sc3)
assert '[URLTAG_0]' in n3, 'url_tag test failed'

print('Normalization smoke tests passed ✓  (ph, x-tags, url-tags)')

## 6 · Load Tokenizer

No `src_lang` / `tgt_lang` needed — opus-mt-tc-big-en-fr is EN→FR only.

In [ ]:
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG

fra_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert fra_token_id != tokenizer.unk_token_id, f'{TGT_LANG} not in vocab'

print(f'Vocab size       : {len(tokenizer):,}')
print(f'fra_Latn token id: {fra_token_id}')

for s in ['[PH_0]', '[BPT_1]', '[URL_0]', '{clock}', '{skip}']:
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f'  {s!r:12} → {len(ids)} token(s)')

## 7 · Load Model in 4-bit + Apply LoRA

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# T4 (Turing) supports FP16 but NOT BF16 — detect at runtime
USE_BF16 = USE_GPU and torch.cuda.is_bf16_supported()
USE_FP16 = USE_GPU and not USE_BF16
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16 if USE_FP16 else torch.float32
print(f'BF16 supported : {USE_BF16}  →  compute dtype: {COMPUTE_DTYPE}')

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
)

if USE_GPU:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    print(f'Loading {MODEL_NAME} in 4-bit (GPU) ...')
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map='auto'
    )
    print(f'GPU mem after load : {torch.cuda.memory_allocated(0)/1e9:.2f} GB')
    # gradient_checkpointing OFF — 600M in 4-bit only uses ~300 MB, T4 has 16 GB free
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
else:
    print(f'Loading {MODEL_NAME} in fp32 (CPU) ...')
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    model.enable_input_require_grads()

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 8 · Tokenize Datasets

In [ ]:
from datasets import Dataset as HFDataset
from transformers import DataCollatorForSeq2Seq

def load_split(path):
    with open(path) as f:
        return HFDataset.from_list(json.load(f))

def preprocess(batch):
    # No src_lang needed — opus-mt is EN→FR only
    return tokenizer(
        text=batch['en'],
        text_target=batch['fr'],
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

print('Loading and tokenizing ...')
train_ds = load_split('data/train.json').map(
    preprocess, batched=True, remove_columns=['en', 'fr'], num_proc=2
)
val_ds = load_split('data/val.json').map(
    preprocess, batched=True, remove_columns=['en', 'fr'], num_proc=2
)
with open('data/test.json') as f:
    test_records = json.load(f)

collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)
print(f'train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_records):,}')
print('Tokenization complete ✓')

## 9 · Fine-Tune

In [ ]:
# ── Pre-training sanity check ─────────────────────────────────────────────────
import sys
errors = []

# Config
if MODEL_NAME != 'facebook/nllb-200-distilled-600M':
    errors.append(f'MODEL_NAME wrong: {MODEL_NAME}')
if MAX_LEN != 64:
    errors.append(f'MAX_LEN should be 64, got {MAX_LEN}')
if 'USE_GPU' not in dir():
    errors.append('USE_GPU not defined — re-run cell 1 (GPU check)')
if 'DEVICE' not in dir():
    errors.append('DEVICE not defined — re-run cell 1')
if 'fra_token_id' not in dir():
    errors.append('fra_token_id not defined — re-run cell 6 (tokenizer)')
if fra_token_id == tokenizer.unk_token_id:
    errors.append('fra_token_id resolved to UNK — tokenizer issue')

# Model
if 'model' not in dir():
    errors.append('model not loaded — re-run cell 7')
else:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable == 0:
        errors.append('No trainable params — LoRA not applied')

# Data
if 'train_ds' not in dir():
    errors.append('train_ds not loaded — re-run cell 8')
if 'val_ds' not in dir():
    errors.append('val_ds not loaded — re-run cell 8')

# Batch vs logit check
logit_gb = TRAIN_BATCH * MAX_LEN * 256_206 * 2 / 1e9
if USE_GPU and logit_gb > 4.0:
    errors.append(f'Logits {logit_gb:.1f} GB may OOM — reduce TRAIN_BATCH')

# Report
if errors:
    print('ERRORS — fix before training:')
    for e in errors: print(f'  ✗ {e}')
    sys.exit(1)
else:
    print('All checks passed ✓')
    print(f'  Model     : {MODEL_NAME}')
    print(f'  Device    : {DEVICE}  (USE_GPU={USE_GPU})')
    print(f'  MAX_LEN   : {MAX_LEN}')
    print(f'  Batch     : {TRAIN_BATCH} × accum {GRAD_ACCUM} = {TRAIN_BATCH*GRAD_ACCUM} effective')
    print(f'  Logits    : {logit_gb:.2f} GB/step')
    print(f'  Trainable : {trainable/1e6:.1f}M params')
    print(f'  fra_token : {fra_token_id}')
    print(f'  train_ds  : {len(train_ds):,} rows')
    print(f'  val_ds    : {len(val_ds):,} rows')

In [ ]:
import time
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, TrainerCallback,
)

class StepLogger(TrainerCallback):
    def __init__(self): self._t0 = time.time()
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            loss = logs.get('loss') or logs.get('eval_loss', '?')
            print(f'[{time.time()-self._t0:6.0f}s | step {state.global_step:5d}] loss={loss}  {logs}', flush=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim='paged_adamw_8bit' if USE_GPU else 'adamw_torch',
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    bf16=USE_BF16,          # True on Ampere+ (L4/A100), False on T4
    fp16=USE_FP16,          # True on T4 (Turing) — native FP16
    gradient_checkpointing=False,   # OFF — 600M in 4-bit fits easily in 16 GB
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,
    logging_steps=50,
    report_to='tensorboard',
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=ES_PATIENCE),
        StepLogger(),
    ],
)

print(f'Starting training  |  device={DEVICE}  bf16={USE_BF16}  fp16={USE_FP16}  grad_ckpt=False')
trainer.train()

## 10 · Save LoRA Adapter

In [ ]:
Path(ADAPTER_DIR).mkdir(exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
adapter_size = sum(p.stat().st_size for p in Path(ADAPTER_DIR).rglob('*') if p.is_file())
print(f'LoRA adapter saved to {ADAPTER_DIR}/  ({adapter_size/1e6:.1f} MB)')

## 11 · Evaluate on Test Set

In [ ]:
import sacrebleu as scb

model.eval()
EVAL_BS = EVAL_BATCH

en_texts = [r['en'] for r in test_records]
fr_refs  = [r['fr'] for r in test_records]
predictions = []

print(f'Translating {len(en_texts):,} test pairs on {DEVICE.upper()} ...')
t_start = time.time()
for i in range(0, len(en_texts), EVAL_BS):
    batch = en_texts[i:i + EVAL_BS]
    inputs = tokenizer(
        batch, return_tensors='pt',
        max_length=MAX_LEN, truncation=True, padding=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=fra_token_id,
            num_beams=4,
            length_penalty=0.6,
            max_new_tokens=MAX_LEN,
        )
    predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    if i % (EVAL_BS * 10) == 0 and i > 0:
        print(f'  {i:,}/{len(en_texts):,}  ({time.time()-t_start:.0f}s)', flush=True)

bleu = scb.corpus_bleu(predictions, [fr_refs])
chrf = scb.corpus_chrf(predictions, [fr_refs], word_order=2)
print(f'\n{"="*52}')
print(f'BLEU   : {bleu.score:.2f}')
print(f'chrF++ : {chrf.score:.2f}')
print(f'{"="*52}')

## 12 · Quick Translate Test

In [ ]:
def translate(text):
    sc = {}
    norm = normalize(text, sc)
    inputs = tokenizer(norm, return_tensors='pt', max_length=MAX_LEN, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=fra_token_id,
            num_beams=4,
            length_penalty=0.6,
            max_new_tokens=MAX_LEN,
        )
    norm_fr = tokenizer.decode(out[0], skip_special_tokens=True)
    if not validate_ph(norm, norm_fr):
        norm_fr = repair_ph(norm, norm_fr)
    return restore(norm_fr, sc)

for s in [
    '{clock} Duration: 90 Mins, {skip} Skip the Line',
    'Explore the Eiffel Tower with a guided tour.',
    'Free cancellation up to 24 hours before the experience.',
    'Mobile tickets accepted. No printout required.',
]:
    print(f'EN: {s}')
    print(f'FR: {translate(s)}')
    print()

## 13 · Export for Backend

Merges LoRA into base model. Final size: **~650 MB** (vs 4.7 GB for NLLB).  
Backend only needs: `pip install transformers torch sentencepiece`

In [ ]:
import shutil, json as _json

EXPORT_DIR = './headout-translator-nllb600'
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
Path(EXPORT_DIR).mkdir()

print('Merging LoRA into base model ...')
t0 = time.time()
merged = model.merge_and_unload()
merged.save_pretrained(EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)
print(f'Merge done in {time.time()-t0:.0f}s')

_json.dump({
    'src_lang': SRC_LANG,
    'tgt_lang': TGT_LANG,
    'fra_token_id': int(fra_token_id),
    'max_new_tokens': MAX_LEN,
    'num_beams': 4,
    'length_penalty': 0.6,
}, open(f'{EXPORT_DIR}/inference_config.json', 'w'), indent=2)

Path(f'{EXPORT_DIR}/translate.py').write_text('''import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class HeadoutTranslator:
    def __init__(self, model_dir: str, device: str = None):
        cfg = json.load(open(Path(model_dir) / "inference_config.json"))
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.tok.src_lang = cfg["src_lang"]
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_dir, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
        ).to(self.device).eval()
        self.cfg = cfg

    def translate(self, text: str) -> str:
        inputs = self.tok(text, return_tensors="pt",
                          max_length=self.cfg["max_new_tokens"], truncation=True).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                forced_bos_token_id=self.cfg["fra_token_id"],
                num_beams=self.cfg["num_beams"],
                length_penalty=self.cfg["length_penalty"],
                max_new_tokens=self.cfg["max_new_tokens"],
            )
        return self.tok.decode(out[0], skip_special_tokens=True)

    def translate_batch(self, texts: list[str]) -> list[str]:
        inputs = self.tok(texts, return_tensors="pt", padding=True,
                          max_length=self.cfg["max_new_tokens"], truncation=True).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                forced_bos_token_id=self.cfg["fra_token_id"],
                num_beams=self.cfg["num_beams"],
                length_penalty=self.cfg["length_penalty"],
                max_new_tokens=self.cfg["max_new_tokens"],
            )
        return self.tok.batch_decode(out, skip_special_tokens=True)

if __name__ == "__main__":
    t = HeadoutTranslator("./headout-translator-nllb600")
    print(t.translate("Skip the Line tickets for the Eiffel Tower."))
''')

total_mb = sum(p.stat().st_size for p in Path(EXPORT_DIR).rglob('*') if p.is_file()) / 1e6
print(f'\nExport: {EXPORT_DIR}  ({total_mb:.0f} MB)')
for p in sorted(Path(EXPORT_DIR).iterdir()):
    if p.is_file():
        print(f'  {p.name:45s} {p.stat().st_size/1e6:6.1f} MB')

print('\nZipping ...')
shutil.make_archive('headout-translator-nllb600', 'zip', '.', EXPORT_DIR)
zip_mb = Path('headout-translator-nllb600.zip').stat().st_size / 1e6
print(f'headout-translator-nllb600.zip  →  {zip_mb:.0f} MB  (single download — no splitting needed)')

try:
    from google.colab import files
    files.download('headout-translator-nllb600.zip')
except ImportError:
    print('Download via Workbench file browser')